In [6]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0)) # 実際に使用されるのは0番目のGPUのみ

True
0
4
NVIDIA RTX A6000


In [3]:
import zipfile
import os

# 展開したいZIPファイルの名前
zip_file_name = 'Yale-Brain-Mets-Longitudinal.zip'

# ZIPファイルが存在するか確認
if not os.path.exists(zip_file_name):
    print(f"エラー: ファイル '{zip_file_name}' が見つかりません。")
else:
    try:
        # ZIPファイルを開き、同じディレクトリに中身をすべて展開する
        with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
            zip_ref.extractall()
        
        print(f"✅ ファイル '{zip_file_name}' の展開が完了しました。")

    except zipfile.BadZipFile:
        print(f"エラー: '{zip_file_name}' は有効なZIPファイルではありません。")
    except Exception as e:
        print(f"展開中に予期せぬエラーが発生しました: {e}")
        

✅ ファイル 'Yale-Brain-Mets-Longitudinal.zip' の展開が完了しました。


In [5]:
import os
import shutil

# --- ここをあなたの環境に合わせて設定してください ---
yale_root_dir = "Yale-Brain-Mets-Longitudinal"# あなたのYaleデータの実際のパス
output_raw_t1post_dir = "my_yale_data/yale_t1post_raw/"
# --- 設定ここまで ---

os.makedirs(output_raw_t1post_dir, exist_ok=True)

target_modality_suffix = "_POST.nii.gz" # T1Post画像を示すサフィックス

# 全患者のディレクトリを走査
for patient_id in os.listdir(yale_root_dir):
    patient_path = os.path.join(yale_root_dir, patient_id)
    if not os.path.isdir(patient_path):
        continue

    # 各日付のディレクトリを走査
    for date_dir in os.listdir(patient_path):
        date_path = os.path.join(patient_path, date_dir)
        if not os.path.isdir(date_path):
            continue

        # 各MRIファイルを走査
        for mri_file in os.listdir(date_path):
            if mri_file.endswith(target_modality_suffix):
                src_path = os.path.join(date_path, mri_file)

                # === ここを修正します ===
                # CASENAMEを生成: PatientID_StudyDate_ScanTime のように一意で簡潔なIDを作成
                # 元のファイル名が YG_0B4NV6E3KEZQ_2015-09-29_12-36-27_POST.nii.gz の場合
                # base_name = YG_0B4NV6E3KEZQ_2015-09-29_12-36-27
                base_name = mri_file.replace(target_modality_suffix, '') # _POST.nii.gz を削除

                # nnUNetが要求する形式 CASENAME_0000.nii.gz にする
                new_filename = f"{base_name}_0000.nii.gz"

                dst_path = os.path.join(output_raw_t1post_dir, new_filename)

                shutil.copy2(src_path, dst_path)
                print(f"Copied {src_path} to {dst_path}")

print(f"\nTotal T1Post images prepared in: {output_raw_t1post_dir}")

Copied Yale-Brain-Mets-Longitudinal/YG_6YBXEK0UYEKW/2014-02-07/YG_6YBXEK0UYEKW_2014-02-07_14-11-42_POST.nii.gz to my_yale_data/yale_t1post_raw/YG_6YBXEK0UYEKW_2014-02-07_14-11-42_0000.nii.gz
Copied Yale-Brain-Mets-Longitudinal/YG_6YBXEK0UYEKW/2013-11-15/YG_6YBXEK0UYEKW_2013-11-15_12-38-31_POST.nii.gz to my_yale_data/yale_t1post_raw/YG_6YBXEK0UYEKW_2013-11-15_12-38-31_0000.nii.gz
Copied Yale-Brain-Mets-Longitudinal/YG_6YBXEK0UYEKW/2014-09-30/YG_6YBXEK0UYEKW_2014-09-30_15-15-01_POST.nii.gz to my_yale_data/yale_t1post_raw/YG_6YBXEK0UYEKW_2014-09-30_15-15-01_0000.nii.gz
Copied Yale-Brain-Mets-Longitudinal/YG_6YBXEK0UYEKW/2014-05-30/YG_6YBXEK0UYEKW_2014-05-30_11-11-24_POST.nii.gz to my_yale_data/yale_t1post_raw/YG_6YBXEK0UYEKW_2014-05-30_11-11-24_0000.nii.gz
Copied Yale-Brain-Mets-Longitudinal/YG_6YBXEK0UYEKW/2012-09-17/YG_6YBXEK0UYEKW_2012-09-17_14-24-54_POST.nii.gz to my_yale_data/yale_t1post_raw/YG_6YBXEK0UYEKW_2012-09-17_14-24-54_0000.nii.gz
Copied Yale-Brain-Mets-Longitudinal/YG_6YBXEK

In [ ]:
import os
import shutil
import SimpleITK as sitk
# nnUNetのpathsモジュールからは何もインポートしません。
# setup_pathsもありません。
# from nnunet.paths import base, preprocessing_output_dir, network_training_output_dir # <-- これらの行を完全に削除！
from nnunet.inference.predict import predict_from_folder 

# --- 環境変数をPythonスクリプト内で設定 ---
# これらのパスは、あなたが実際に作成したディレクトリの絶対パスに置き換えてください。
os.environ['nnUNet_raw_data_base'] = "/home/sys/s276255/notebooks/nnUNet_data/nnUNet_raw_data_base"
os.environ['nnUNet_preprocessed'] = "/home/sys/s276255/notebooks/nnUNet_data/nnUNet_preprocessed"
os.environ['RESULTS_FOLDER'] = "/home/sys/s276255/notebooks/nnUNet_results" 

print("nnUNet environment variables set within script.")

# --- 環境変数が正しく設定されたか確認 ---
# これらはNone以外（設定したパス）が表示されるはずです。
print(f"nnUNet_raw_data_base (from os.environ): {os.environ.get('nnUNet_raw_data_base')}") 
print(f"nnUNet_preprocessed (from os.environ): {os.environ.get('nnUNet_preprocessed')}")
print(f"RESULTS_FOLDER (from os.environ): {os.environ.get('RESULTS_FOLDER')}")
# nnunet.pathsからのPython変数はもう使わないので、ここも表示不要です
# print(f"nnUNet_raw_data (Python var from paths.py): {base}")
# print(f"preprocessing_output_dir (Python var from paths.py): {preprocessing_output_dir}")
# print(f"network_training_output_dir (Python var from paths.py): {network_training_output_dir}")


output_raw_t1post_dir = "my_yale_data/yale_t1post_raw/"
# --- ステップ3.3: 頭蓋骨除去の実行 ---
output_skullstripped_dir = "/home/sys/s276255/notebooks/my_yale_data/yale_t1post_skullstripped"
os.makedirs(output_skullstripped_dir, exist_ok=True)

plans_pkl_path = os.path.join(output_skullstripped_dir, "plans.pkl")
if os.path.exists(plans_pkl_path):
    os.remove(plans_pkl_path)
    print(f"Removed existing plans.pkl from {output_skullstripped_dir}")


# 頭蓋骨除去モデルの設定
task_name = "Task105_T1postskullstrip" 
trainer_class_name = "nnUNetTrainerV2"
plans_identifier = "nnUNetPlansv2.1" 

configuration_name = "3d_fullres" 

folds = "all" 
save_npz = False 
num_threads_preprocessing = 96 
num_threads_nifti_save = 96 
tta = False 
overwrite_existing = False

lowres_segmentations = None 
part_id = 0 
num_parts = 1 


# モデルフォルダのフルパスを構築 (predict_from_folderの'model'引数に渡す値)
# network_training_output_dir の代わりに os.environ.get('RESULTS_FOLDER') を使用します。
model_path_for_function = os.path.join(os.environ.get('RESULTS_FOLDER'), 
                                 "nnUNet", # RESULTS_FOLDERの直下にあるnnUNetフォルダ
                                 configuration_name, 
                                 task_name, 
                                 trainer_class_name + "__" + plans_identifier)

print(f"\nAttempting to use skull stripping model from: {model_path_for_function}")

# nnUNetの予測関数を実行
try:
    predict_from_folder(
        model=model_path_for_function, # 'model'引数にパス全体を渡す
        input_folder=output_raw_t1post_dir, 
        output_folder=output_skullstripped_dir, 
        folds=folds,
        save_npz=save_npz,
        num_threads_preprocessing=num_threads_preprocessing,
        num_threads_nifti_save=num_threads_nifti_save,
        tta=tta, 
        overwrite_existing=overwrite_existing,
        lowres_segmentations=lowres_segmentations,
        part_id=part_id,
        num_parts=num_parts
    )
    print("\nSkull stripping prediction completed successfully!")

except Exception as e:
    print(f"\nAn error occurred during skull stripping prediction: {e}")
    print("Please double check:")
    print(f"1. Is the model folder actually present at: {model_path_for_function}")
    print("   (You can check with 'ls -l' in a terminal)")
    print(f"2. Are the input files in '{output_raw_t1post_dir}' named correctly (CASENAME_0000.nii.gz)?")
    print("3. Check for any other specific error messages from nnUNet, not just generic Python errors.")

nnUNet environment variables set within script.
nnUNet_raw_data_base (from os.environ): /home/sys/s276255/notebooks/nnUNet_data/nnUNet_raw_data_base
nnUNet_preprocessed (from os.environ): /home/sys/s276255/notebooks/nnUNet_data/nnUNet_preprocessed
RESULTS_FOLDER (from os.environ): /home/sys/s276255/notebooks/nnUNet_results

Attempting to use skull stripping model from: /home/sys/s276255/notebooks/nnUNet_results/nnUNet/3d_fullres/Task105_T1postskullstrip/nnUNetTrainerV2__nnUNetPlansv2.1
This model expects 1 input modalities for each image
Found 8996 unique case ids, here are some examples: ['YG_GOFWB2AUALLF_2017-09-11_10-08-25'
 'YG_UMCVCZO0AU1F_2022-08-23_19-55-24'
 'YG_RBK95BX60SKH_2016-10-14_13-29-37'
 'YG_5LFL4U4G8GPT_2014-01-19_11-15-38'
 'YG_7F6DV1EVTU48_2021-11-25_16-02-12'
 'YG_AZH5KZFA9IKT_2019-01-23_08-38-54'
 'YG_MSYA9UP2GCZ4_2020-11-09_10-50-13'
 'YG_TGGSFW380L05_2012-05-12_17-11-18'
 'YG_4FWHY1BHBX60_2021-08-18_09-12-48'
 'YG_8H3GW3R6T0DJ_2022-10-28_09-01-48']
If they don't

In [5]:
import os
import shutil
import subprocess
import glob

# --- パス設定（以前のスクリプトと同じパスを使用） ---
yale_root_dir = "/path/to/Yale-Brain-Mets-Longitudinal" # Yaleデータセットのルートディレクトリ
output_raw_t1post_dir = "/home/sys/s276255/notebooks/my_yale_data/yale_t1post_raw" # nnUNet形式に変換した生データ保存先
output_skullstripped_dir = "/home/sys/s276255/notebooks/my_yale_data/yale_t1post_skullstripped" # 頭蓋骨除去後の出力先

# nnUNet環境変数の設定（スクリプトの冒頭で一度だけ実行）
os.environ['nnUNet_raw_data_base'] = '/home/sys/s276255/notebooks/nnUNet_data/nnUNet_raw_data_base'
os.environ['nnUNet_preprocessed'] = '/home/sys/s276255/notebooks/nnUNet_data/nnUNet_preprocessed'
os.environ['RESULTS_FOLDER'] = '/home/sys/s276255/notebooks/nnUNet_results'
print("nnUNet environment variables set within script.")
print(f"nnUNet_raw_data_base (from os.environ): {os.environ.get('nnUNet_raw_data_base')}")
print(f"nnUNet_preprocessed (from os.environ): {os.environ.get('nnUNet_preprocessed')}")
print(f"RESULTS_FOLDER (from os.environ): {os.environ.get('RESULTS_FOLDER')}")

# --- メイン処理 ---
target_modality_suffix = "_POST.nii.gz"
batch_size = 100 # 一度に処理する画像の枚数。ここを調整してください。

# 1. 全ての生データファイルを取得
all_raw_files = [f for f in os.listdir(output_raw_t1post_dir) if f.endswith('_0000.nii.gz')]
all_raw_case_ids = {os.path.basename(f).replace('_0000.nii.gz', '') for f in all_raw_files}

# 2. 既に処理済みのファイルを取得
# nnUNetの出力は通常、元のファイル名と同じになるはずです
processed_files = [f for f in os.listdir(output_skullstripped_dir) if f.endswith('.nii.gz')]
processed_case_ids = {os.path.basename(f).replace('.nii.gz', '') for f in processed_files}

# 3. 未処理のケースIDを特定
unprocessed_case_ids = sorted(list(all_raw_case_ids - processed_case_ids))
print(f"Total raw files found: {len(all_raw_files)}")
print(f"Total processed files found: {len(processed_files)}")
print(f"Total unprocessed cases remaining: {len(unprocessed_case_ids)}")

if not unprocessed_case_ids:
    print("All cases have already been skull stripped. Exiting.")
else:
    # バッチ処理用のディレクトリを作成
    batch_input_dir = os.path.join(output_raw_t1post_dir, "current_batch_input")
    os.makedirs(batch_input_dir, exist_ok=True)

    for i in range(0, len(unprocessed_case_ids), batch_size):
        current_batch_case_ids = unprocessed_case_ids[i:i + batch_size]
        print(f"\n--- Processing batch {i // batch_size + 1} of {len(unprocessed_case_ids) // batch_size + 1} ({len(current_batch_case_ids)} images) ---")

        # 既存のバッチ入力ディレクトリをクリア
        for f in os.listdir(batch_input_dir):
            os.remove(os.path.join(batch_input_dir, f))

        # 現在のバッチのファイルを一時ディレクトリにコピー
        for case_id in current_batch_case_ids:
            src_file = os.path.join(output_raw_t1post_dir, f"{case_id}_0000.nii.gz")
            dst_file = os.path.join(batch_input_dir, f"{case_id}_0000.nii.gz")
            if os.path.exists(src_file):
                shutil.copy2(src_file, dst_file)
            else:
                print(f"Warning: Source file not found for {case_id}: {src_file}")

        # nnUNet_predict コマンドの実行
        # 注意: ここでは nnUNet_predict コマンドを直接呼び出します
        # nnUNet_predict -i <input_folder> -o <output_folder> -t <TaskID> -m <model_type> -f <fold(s)> --disable_tta --num_threads_preprocessing <threads> --num_threads_nifti_save <threads>
        command = [
            "nnUNet_predict",
            "-i", batch_input_dir,
            "-o", output_skullstripped_dir, # 出力は最終的なディレクトリに直接保存
            "-t", "Task105_T1postskullstrip",
            "-m", "3d_fullres",
            "-f", "all",
            "--disable_tta", # 推論時間を短縮するため、TTAを無効化
            "--num_threads_preprocessing", "96", # CPUスレッド数
            "--num_threads_nifti_save", "12" # NIfTI保存スレッド数
        ]

        try:
            print(f"Executing command: {' '.join(command)}")
            # subprocess.run を使用してコマンドを実行し、出力をリアルタイムで表示
            process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                print(line, end='')
            process.wait()

            if process.returncode != 0:
                print(f"Error in nnUNet_predict for batch starting with {current_batch_case_ids[0]}. Return code: {process.returncode}")
                # エラーが発生した場合、ここで処理を停止するか、ログを記録して続行するか選択
                # 例: break # 停止する場合
            else:
                print(f"Batch {i // batch_size + 1} completed successfully.")

        except Exception as e:
            print(f"An error occurred during batch processing: {e}")
            # エラーが発生した場合の処理
            break # 処理を停止

    print("\n--- All available batches processed. ---")

nnUNet environment variables set within script.
nnUNet_raw_data_base (from os.environ): /home/sys/s276255/notebooks/nnUNet_data/nnUNet_raw_data_base
nnUNet_preprocessed (from os.environ): /home/sys/s276255/notebooks/nnUNet_data/nnUNet_preprocessed
RESULTS_FOLDER (from os.environ): /home/sys/s276255/notebooks/nnUNet_results
Total raw files found: 8996
Total processed files found: 8996
Total unprocessed cases remaining: 0
All cases have already been skull stripped. Exiting.


In [11]:
import os
import shutil

# --- パス設定 ---
# 頭蓋骨除去後の出力ディレクトリ（ここにあるファイル名を修正する）
skullstripped_input_dir = "/home/sys/s276255/notebooks/my_yale_data/yale_t1post_skullstripped" 

print(f"Renaming files in {skullstripped_input_dir} to nnUNet format (CASENAME_0000.nii.gz)...")

for filename in os.listdir(skullstripped_input_dir):
    if filename.endswith(".nii.gz"):
        src_path = os.path.join(skullstripped_input_dir, filename)

        # 元のファイル名から拡張子を除去
        base_name = filename.replace(".nii.gz", "")

        # ファイル名が既に _0000 の形式になっていないかチェック
        if base_name.endswith("_0000"):
            # 既に正しい形式であれば何もしない
            new_filename = filename
        else:
            # 正しい形式に修正
            new_filename = f"{base_name}_0000.nii.gz"

        dst_path = os.path.join(skullstripped_input_dir, new_filename)

        if src_path != dst_path: # 同じファイル名でなければリネーム
            try:
                os.rename(src_path, dst_path)
                print(f"Renamed: {filename} -> {new_filename}")
            except OSError as e:
                print(f"Error renaming {filename} to {new_filename}: {e}")
        else:
            print(f"Skipped: {filename} (already in correct format or no change needed)")

print("File renaming completed.")

Renaming files in /home/sys/s276255/notebooks/my_yale_data/yale_t1post_skullstripped to nnUNet format (CASENAME_0000.nii.gz)...
Skipped: YG_AX7KKXEY6L6P_2018-06-30_08-23-40_0000.nii.gz (already in correct format or no change needed)
Skipped: YG_A5PHRABC9LL0_2022-03-01_09-00-38_0000.nii.gz (already in correct format or no change needed)
Skipped: YG_TLJGEX9O8IIN_2021-08-31_16-55-24_0000.nii.gz (already in correct format or no change needed)
Skipped: YG_QU86MZOUT0VE_2015-07-06_21-41-10_0000.nii.gz (already in correct format or no change needed)
Skipped: YG_X5JM75SM8T3A_2018-07-07_13-37-59_0000.nii.gz (already in correct format or no change needed)
Skipped: YG_I9HD9FQ15DCI_2012-09-27_12-04-20_0000.nii.gz (already in correct format or no change needed)
Skipped: YG_VV13QFMENASZ_2014-12-21_12-05-53_0000.nii.gz (already in correct format or no change needed)
Skipped: YG_WT30WSLBXDLO_2013-06-09_14-10-19_0000.nii.gz (already in correct format or no change needed)
Skipped: YG_FBS0P6I3MHT2_2021-01

In [ ]:
import os
import shutil
import SimpleITK as sitk
# nnUNetのpathsモジュールからは何もインポートしません。
# setup_pathsもありません。
# from nnunet.paths import base, preprocessing_output_dir, network_training_output_dir # <-- これらの行を完全に削除！
from nnunet.inference.predict import predict_from_folder 

# --- 環境変数をPythonスクリプト内で設定 ---
# これらのパスは、あなたが実際に作成したディレクトリの絶対パスに置き換えてください。
os.environ['nnUNet_raw_data_base'] = "/home/sys/s276255/notebooks/nnUNet_data/nnUNet_raw_data_base"
os.environ['nnUNet_preprocessed'] = "/home/sys/s276255/notebooks/nnUNet_data/nnUNet_preprocessed"
os.environ['RESULTS_FOLDER'] = "/home/sys/s276255/notebooks/nnUNet_results" 

print("nnUNet environment variables set within script.")

# --- 環境変数が正しく設定されたか確認 ---
# これらはNone以外（設定したパス）が表示されるはずです。
print(f"nnUNet_raw_data_base (from os.environ): {os.environ.get('nnUNet_raw_data_base')}") 
print(f"nnUNet_preprocessed (from os.environ): {os.environ.get('nnUNet_preprocessed')}")
print(f"RESULTS_FOLDER (from os.environ): {os.environ.get('RESULTS_FOLDER')}")
# nnunet.pathsからのPython変数を使わないので、ここでは表示しません。


# --- ここからは脳転移セグメンテーション用の設定 ---
# 入力ディレクトリ（頭蓋骨除去済みのYale T1Post画像がある場所）
input_folder_for_brain_mets = "/home/sys/s276255/notebooks/my_yale_data/yale_t1post_skullstripped"

# 出力ディレクトリ（脳転移セグメンテーション結果が保存される場所）
output_brain_mets_segmentations_dir = "/home/sys/s276255/notebooks/my_yale_data/yale_brain_mets_segmentations"
os.makedirs(output_brain_mets_segmentations_dir, exist_ok=True)


# 念のため、出力ディレクトリに既存のplans.pklがあれば削除
plans_pkl_path_mets = os.path.join(output_brain_mets_segmentations_dir, "plans.pkl")
if os.path.exists(plans_pkl_path_mets):
    os.remove(plans_pkl_path_mets)
    print(f"Removed existing plans.pkl from {output_brain_mets_segmentations_dir}")


# 脳転移セグメンテーションモデルの設定
task_name_mets = "Task117_Metastases_Post" 
trainer_class_name_mets = "nnUNetTrainerV2"
plans_identifier_mets = "nnUNetPlansv2.1" 

configuration_name_mets = "3d_fullres" 

folds = "all" 
save_npz = False 
num_threads_preprocessing = 96 
num_threads_nifti_save = 96 
tta = False 
overwrite_existing = False

lowres_segmentations = None 
part_id = 0 
num_parts = 1 


# モデルフォルダのフルパスを構築 (predict_from_folderの'model'引数に渡す値)
# os.environ.get('RESULTS_FOLDER')から直接パスを取得し、nnUNetの構造に合わせて追加
model_path_for_function_mets = os.path.join(os.environ.get('RESULTS_FOLDER'), 
                                 "nnUNet", # RESULTS_FOLDERの直下にあるnnUNetフォルダ
                                 configuration_name_mets, 
                                 task_name_mets, 
                                 trainer_class_name_mets + "__" + plans_identifier_mets)

print(f"\nAttempting to use brain metastasis segmentation model from: {model_path_for_function_mets}")

# nnUNetの予測関数を実行
try:
    predict_from_folder(
        model=model_path_for_function_mets, # 'model'引数にパス全体を渡す
        input_folder=input_folder_for_brain_mets, 
        output_folder=output_brain_mets_segmentations_dir, 
        folds=folds,
        save_npz=save_npz,
        num_threads_preprocessing=num_threads_preprocessing,
        num_threads_nifti_save=num_threads_nifti_save,
        tta=tta, 
        overwrite_existing=overwrite_existing,
        lowres_segmentations=lowres_segmentations,
        part_id=part_id,
        num_parts=num_parts
    )
    print("\nBrain metastasis segmentation prediction completed successfully!")

except Exception as e:
    print(f"\nAn error occurred during brain metastasis segmentation prediction: {e}")
    print("Please double check:")
    print(f"1. Is the model folder actually present at: {model_path_for_function_mets}")
    print("   (You can check with 'ls -l' in a terminal)")
    print(f"2. Are the input files in '{input_folder_for_brain_mets}' correctly processed (skull-stripped) and named (CASENAME_0000.nii.gz)?")
    print("3. Check for any other specific error messages from nnUNet.")



Please cite the following paper when using nnUNet:

Isensee, F., Jaeger, P.F., Kohl, S.A.A. et al. "nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation." Nat Methods (2020). https://doi.org/10.1038/s41592-020-01008-z


If you have questions or suggestions, feel free to open an issue at https://github.com/MIC-DKFZ/nnUNet

nnUNet_raw_data_base is not defined and nnU-Net can only be used on data for which preprocessed files are already present on your system. nnU-Net cannot be used for experiment planning and preprocessing like this. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up properly.
nnUNet_preprocessed is not defined and nnU-Net can not be used for preprocessing or training. If this is not intended, please read documentation/setting_up_paths.md for information on how to set this up.
RESULTS_FOLDER is not defined and nnU-Net cannot be used for training or inference. If this is not in

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0)) # 実際に使用されるのは0番目のGPUのみ

True
0
4
NVIDIA RTX A6000


In [ ]:
import os
import shutil
import SimpleITK as sitk
# nnUNetのpathsモジュールからは何もインポートしません。
# setup_pathsもありません。
# from nnunet.paths import base, preprocessing_output_dir, network_training_output_dir # <-- これらの行を完全に削除！
from nnunet.inference.predict import predict_from_folder 

# --- 環境変数をPythonスクリプト内で設定 ---
# これらのパスは、あなたが実際に作成したディレクトリの絶対パスに置き換えてください。
os.environ['nnUNet_raw_data_base'] = "/home/sys/s276255/notebooks/nnUNet_data/nnUNet_raw_data_base"
os.environ['nnUNet_preprocessed'] = "/home/sys/s276255/notebooks/nnUNet_data/nnUNet_preprocessed"
os.environ['RESULTS_FOLDER'] = "/home/sys/s276255/notebooks/nnUNet_results" 

print("nnUNet environment variables set within script.")

# --- 環境変数が正しく設定されたか確認 ---
# これらはNone以外（設定したパス）が表示されるはずです。
print(f"nnUNet_raw_data_base (from os.environ): {os.environ.get('nnUNet_raw_data_base')}") 
print(f"nnUNet_preprocessed (from os.environ): {os.environ.get('nnUNet_preprocessed')}")
print(f"RESULTS_FOLDER (from os.environ): {os.environ.get('RESULTS_FOLDER')}")


# --- 脳転移セグメンテーション用の設定 ---
# 入力ディレクトリ（ここが最重要！元のYale画像で、_0000.nii.gzにリネーム済みのもの）
# 前回「yale_t1post_raw」にコピーしたファイルを使用します
input_folder_for_brain_mets = "/home/sys/s276255/notebooks/my_yale_data/yale_t1post_raw"

# 出力ディレクトリ（脳転移セグメンテーション結果が保存される場所）
output_brain_mets_segmentations_dir = "/home/sys/s276255/notebooks/my_yale_data/yale_brain_mets_segmentations"
os.makedirs(output_brain_mets_segmentations_dir, exist_ok=True)


# 念のため、出力ディレクトリに既存のplans.pklがあれば削除
plans_pkl_path_mets = os.path.join(output_brain_mets_segmentations_dir, "plans.pkl")
if os.path.exists(plans_pkl_path_mets):
    os.remove(plans_pkl_path_mets)
    print(f"Removed existing plans.pkl from {output_brain_mets_segmentations_dir}")


# 脳転移セグメンテーションモデルの設定
task_name_mets = "Task117_Metastases_Post" 
trainer_class_name_mets = "nnUNetTrainerV2"
plans_identifier_mets = "nnUNetPlansv2.1" 

configuration_name_mets = "3d_fullres" 

folds = "all" 
save_npz = False 
num_threads_preprocessing = 32 # スレッド数を減らしています
num_threads_nifti_save = 32   # スレッド数を減らしています
tta = False 
overwrite_existing = False

lowres_segmentations = None 
part_id = 0 
num_parts = 1 


# モデルフォルダのフルパスを構築 (predict_from_folderの'model'引数に渡す値)
# os.environ.get('RESULTS_FOLDER')から直接パスを取得し、nnUNetの構造に合わせて追加
model_path_for_function_mets = os.path.join(os.environ.get('RESULTS_FOLDER'), # <-- ここを直接 os.environ.get() に修正
                                 "nnUNet", # RESULTS_FOLDERの直下にあるnnUNetフォルダ
                                 configuration_name_mets, 
                                 task_name_mets, 
                                 trainer_class_name_mets + "__" + plans_identifier_mets)

print(f"\nAttempting to use brain metastasis segmentation model from: {model_path_for_function_mets}")

# nnUNetの予測関数を実行
try:
    predict_from_folder(
        model=model_path_for_function_mets, # 'model'引数にパス全体を渡す
        input_folder=input_folder_for_brain_mets, 
        output_folder=output_brain_mets_segmentations_dir, 
        folds=folds,
        save_npz=save_npz,
        num_threads_preprocessing=num_threads_preprocessing,
        num_threads_nifti_save=num_threads_nifti_save,
        tta=tta, 
        overwrite_existing=overwrite_existing,
        lowres_segmentations=lowres_segmentations,
        part_id=part_id,
        num_parts=num_parts
    )
    print("\nBrain metastasis segmentation prediction completed successfully!")

except Exception as e:
    print(f"\nAn error occurred during brain metastasis segmentation prediction: {e}")
    print("Please double check:")
    print(f"1. Is the model folder actually present at: {model_path_for_function_mets}")
    print("   (You can check with 'ls -l' in a terminal)")
    print(f"2. Are the input files in '{input_folder_for_brain_mets}' correctly named (CASENAME_0000.nii.gz) and **contain actual brain data (not all black)**?") 
    print("3. Check for any other specific error messages from nnUNet.")

nnUNet environment variables set within script.
nnUNet_raw_data_base (from os.environ): /home/sys/s276255/notebooks/nnUNet_data/nnUNet_raw_data_base
nnUNet_preprocessed (from os.environ): /home/sys/s276255/notebooks/nnUNet_data/nnUNet_preprocessed
RESULTS_FOLDER (from os.environ): /home/sys/s276255/notebooks/nnUNet_results

Attempting to use brain metastasis segmentation model from: /home/sys/s276255/notebooks/nnUNet_results/nnUNet/3d_fullres/Task117_Metastases_Post/nnUNetTrainerV2__nnUNetPlansv2.1
This model expects 1 input modalities for each image
Found 8996 unique case ids, here are some examples: ['YG_KB4AMH3NXCV2_2013-04-13_10-15-25'
 'YG_W5YYQ5QICNMX_2021-07-03_09-55-36'
 'YG_MV491BMDPUNJ_2022-02-28_19-55-02'
 'YG_37PXRCQ2J3UK_2013-11-22_07-26-27'
 'YG_QBTO19I1R78V_2019-08-09_08-36-52'
 'YG_NRSFSRTTONBY_2016-08-07_16-00-40'
 'YG_Q7D6O0EB0GA5_2015-11-16_13-29-41'
 'YG_4G2PRLW3R8N8_2022-05-31_07-55-38'
 'YG_S9ZFLB35NW0T_2015-02-22_12-03-23'
 'YG_C9D2TEGNY08A_2012-07-31_07-22-39']


/home/sys/s276255/notebooks/nnUNet/nnunet/training/network_training/network_trainer.py:404: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.amp_grad_scaler = GradScaler()
/home/sys/s276255/notebooks/nnUNet/nnunet/network_architecture/neural_network.py:141: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with context():


separate z, order in z is 0separate z, order in z is  order inplane is0  1order inplane is
 1
before crop: (1, 112, 256, 256) after crop: before crop:(1, 68, 159, 130)  (1, 32, 512, 512)spacing:  after crop: [2. 1. 1.](1, 28, 407, 334)  
spacing:
 [4.9999876  0.42969999 0.42969996] 

no separate z, order 3before crop:
 (1, 32, 512, 504) after crop:before crop:  (1, 28, 361, 288) (1, 88, 256, 256)spacing:  after crop:[5.         0.44921875 0.44921878]  (1, 69, 170, 133)
 
spacing: [2. 1. 1.] 

separate z, order in z is 0 order inplane is 1
before:no separate z, order  3{'spacing': array([5.       , 0.8984375, 0.8984375]), 'spacing_transposed': array([5.       , 0.8984375, 0.8984375]), 'data.shape (data is transposed)': (1, 28, 199, 158)} 

 {'spacing': array([1.5       , 0.71428573, 0.71428573]), 'data.shape (data is resampled)': (1, 93, 250, 199)} 

separate z, order in z is 0 order inplane is 3separate z, order in z is
 0 separate z, order in z isorder inplane is  01 
order inplane is

In [1]:
import os
import nibabel as nib
import numpy as np

# --- 設定（あなたの環境に合わせてください） ---
# 予測対象の全画像数
expected_total_images = 8996 

# 脳転移セグメンテーション結果の出力ディレクトリ
output_brain_mets_segmentations_dir = "/home/sys/s276255/notebooks/my_yale_data/yale_brain_mets_segmentations"
# --- 設定ここまで ---

print("--- nnUNet Prediction Completion Check ---")

# 1. 出力ディレクトリのファイル数を確認
print(f"\nChecking output directory: {output_brain_mets_segmentations_dir}")
generated_files = [f for f in os.listdir(output_brain_mets_segmentations_dir) if f.endswith('.nii.gz')]
num_generated_files = len(generated_files)

print(f"Expected number of output files: {expected_total_images}")
print(f"Found number of generated files: {num_generated_files}")

if num_generated_files == expected_total_images:
    print("\n✅ **SUCCESS**: All expected files have been generated! The prediction process likely completed.")
else:
    print(f"\n⚠️ **WARNING**: Number of generated files ({num_generated_files}) does not match expected total ({expected_total_images}).")
    if num_generated_files > 0:
        print(f"   {expected_total_images - num_generated_files} files are still missing or the process was interrupted.")
    else:
        print("   No output files were generated. The prediction process may not have started or failed immediately.")

# 2. 生成されたファイルの簡易的な内容チェック（真っ黒でないか）
print("\nPerforming quick check on generated file content (first 10 files)...")
checked_count = 0
black_image_count = 0

for i, filename in enumerate(generated_files):
    if checked_count >= 10: # 最初の10ファイルのみチェック
        break

    file_path = os.path.join(output_brain_mets_segmentations_dir, filename)
    
    try:
        img = nib.load(file_path)
        data = img.get_fdata()
        
        # ボクセル値の平均を計算（脳組織や病変があれば0より大きいはず）
        # nnUNetのセグメンテーションマスクは通常0か1なので、平均が0なら真っ黒
        data_mean = np.mean(data)
        
        if data_mean == 0:
            black_image_count += 1
            print(f"  - {filename}: Appears to be all black (mean value: {data_mean:.4f})")
        else:
            print(f"  - {filename}: OK (mean value: {data_mean:.4f})")
            
    except Exception as e:
        print(f"  - Error loading or processing {filename}: {e}")
    
    checked_count += 1

if black_image_count > 0:
    print(f"\n❌ **PROBLEM**: {black_image_count} of the checked files appear to be all black. This indicates a potential issue with the segmentation output (e.g., model failed to find anything, or output is corrupted).")
    print("   Please check these files visually in 3D Slicer.")
elif num_generated_files > 0:
    print("\n✅ **OK**: Checked files do not appear to be all black. This is a good sign.")
    print("   However, visual inspection in 3D Slicer is still highly recommended for quality assessment.")
else:
    print("\nNo files to check content for (no files generated yet).")

print("\n--- Check Complete ---")
print(f"Please examine files in {output_brain_mets_segmentations_dir} visually using 3D Slicer or ITK-SNAP.")

--- nnUNet Prediction Completion Check ---

Checking output directory: /home/sys/s276255/notebooks/my_yale_data/yale_brain_mets_segmentations
Expected number of output files: 8996
Found number of generated files: 8996

✅ **SUCCESS**: All expected files have been generated! The prediction process likely completed.

Performing quick check on generated file content (first 10 files)...
  - YG_7828CY2JYO5N_2019-04-13_18-28-06.nii.gz: OK (mean value: 0.0006)
  - YG_9F0K71ZPLCT6_2012-12-11_11-52-47.nii.gz: OK (mean value: 0.0002)
  - YG_2XJ5E15EJJ87_2019-10-11_09-02-55.nii.gz: OK (mean value: 0.0002)
  - YG_BL9FMVUR8I8K_2017-07-30_11-30-17.nii.gz: Appears to be all black (mean value: 0.0000)
  - YG_VV13QFMENASZ_2012-10-07_11-45-59.nii.gz: OK (mean value: 0.0000)
  - YG_8ACYU6H9VJX3_2015-06-17_10-40-14.nii.gz: OK (mean value: 0.0018)
  - YG_FBC9ENDNLFQ3_2013-07-13_10-25-45.nii.gz: OK (mean value: 0.0004)
  - YG_OKSZI2CG9F2A_2017-11-12_15-36-35.nii.gz: OK (mean value: 0.0022)
  - YG_31QQSUJFPF7

In [ ]:
# 大学サーバのファイルからローカルにファイルを移しておく
import os
import zipfile

# 圧縮するディレクトリを指定します
dir_to_zip = './' 
# 作成するZIPファイルの名前を指定します
zip_file_name = 'all_notebooks.zip'

# 圧縮処理を実行する関数
def zip_directory(path, ziph):
    for root, dirs, files in os.walk(path):
        for file in files:
            # ZIPファイル自体を圧縮対象から除外します
            if file == zip_file_name:
                continue
            # ファイルの絶対パスを取得
            file_path = os.path.join(root, file)
            # ZIPファイル内のパス（ディレクトリ構造を保持するため）
            # './' を基準にした相対パスに変換
            arcname = os.path.relpath(file_path, dir_to_zip)
            print(f'Compressing: {file_path} as {arcname}')
            ziph.write(file_path, arcname)

# zipファイルを書き込みモードで開く
with zipfile.ZipFile(zip_file_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zip_directory(dir_to_zip, zipf)

print(f"'{zip_file_name}' was successfully created!")

Compressing: ./preprocess.ipynb as preprocess.ipynb
Compressing: ./skullstrip_FLAIR.zip as skullstrip_FLAIR.zip
Compressing: ./Metastases_T1post_T1pre.zip as Metastases_T1post_T1pre.zip
Compressing: ./PretrainedNetworksforMetastasesPaper.tar.gz as PretrainedNetworksforMetastasesPaper.tar.gz
Compressing: ./T1post_Skullstrip.zip as T1post_Skullstrip.zip
Compressing: ./Metastases_T1post_T1pre_FLAIR.zip as Metastases_T1post_T1pre_FLAIR.zip
Compressing: ./Metastases_Subtraction.zip as Metastases_Subtraction.zip
